# Model training

## 1. How the model works

- **TF-IDF** converts text into numeric features based on how informative words or character patterns are.
- **Word n-grams** capture words and short word sequences.
- **Character n-grams** can capture spelling variants, partial words, and obfuscated abusive language.
- **Logistic Regression** estimates class probabilities.
- **OneVsRestClassifier** trains one binary classifier per toxicity label.
- **Threshold tuning** chooses a cutoff on the validation set, not the test set.
- **F1** balances precision and recall; the script tunes per-label toxicity F1 and spam F1.

In [1]:
from pathlib import Path
import subprocess
import sys

ROOT = Path.cwd().resolve()
if not (ROOT / "src" / "train_models.py").is_file() and (ROOT.parent / "src" / "train_models.py").is_file():
    ROOT = ROOT.parent
SCRIPT = ROOT / "src" / "train_models.py"
if not SCRIPT.is_file():
    raise FileNotFoundError(f"Cannot find {SCRIPT}. Open this notebook from the project or its notebooks folder.")
print(f"Project root: {ROOT}")
print(f"Training script: {SCRIPT}")

Project root: /Users/kakulrajput/Documents/Community_App_ML
Training script: /Users/kakulrajput/Documents/Community_App_ML/src/train_models.py


##  training 

In [2]:
result = subprocess.run(
    [sys.executable, str(SCRIPT)],
    cwd=ROOT,
    text=True,
    capture_output=True,
)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError("Training script failed.")


Spam model comparison on validation data:
                 model  threshold       f2       f1  precision  recall
   logistic_regression       0.41 0.971787 0.976378   0.984127 0.96875
complement_naive_bayes       0.61 0.946372 0.960000   0.983607 0.93750
Selected spam model: logistic_regression
Spam examples:
                                                                       text  spam_probability  threshold predicted_label
                                                        Get some free money          0.303406       0.41             ham
CONGRATULATIONS! You won a free prize! Click here now to claim your reward!          0.971065       0.41            spam
                                         Hey, are we still meeting at 6 pm?          0.016902       0.41             ham
                    Thank you for helping our community organize the event.          0.175410       0.41             ham
Training finished. Reports and thresholds are saved in artifacts/.
Toxicity macro F1

## 3. Inspect saved results

In [3]:
import json
import pandas as pd
from IPython.display import display

ARTIFACTS = ROOT / "artifacts"
expected = ["toxicity_test_report.json", "sms_spam_test_report.json", "toxicity_word_char_tfidf.joblib", "sms_spam_tfidf_logistic_regression.joblib"]
missing = [name for name in expected if not (ARTIFACTS / name).is_file()]
if missing:
    print("Missing outputs:", missing)
else:
    toxicity = json.loads((ARTIFACTS / "toxicity_test_report.json").read_text(encoding="utf-8"))
    spam = json.loads((ARTIFACTS / "sms_spam_test_report.json").read_text(encoding="utf-8"))
    display(pd.DataFrame([
        {"model": "Toxicity", "macro_f1": toxicity.get("macro_f1"), "micro_f1": toxicity.get("micro_f1"), "test_rows": toxicity.get("test_rows")},
        {"model": "SMS spam", "accuracy": spam.get("accuracy"), "spam_f1": spam.get("spam_f1"), "test_rows": spam.get("test_rows")},
    ]))

,model,macro_f1,micro_f1,test_rows,accuracy,spam_f1
0,Toxicity,0.632712,0.766417,15958,NaN,NaN
1,SMS spam,NaN,NaN,516,0.988372,0.953125


## 4. Output files

- `artifacts/toxicity_word_char_tfidf.joblib` — fitted toxicity feature transformers, model, labels, thresholds.
- `artifacts/sms_spam_tfidf_logistic_regression.joblib` — fitted SMS feature transformers, model, threshold.
- `artifacts/toxicity_test_report.json` — multilabel toxicity metrics and per-label classification report.
- `artifacts/sms_spam_test_report.json` — spam metrics, classification report, and confusion matrix.
- `artifacts/toxicity_thresholds.json` and `artifacts/sms_spam_threshold.json` — thresholds chosen on validation data.

## 5. Verify spam model on practical examples
The training script compares Logistic Regression and Complement Naive Bayes using validation F2, saves the selected model using the existing artifact filename, and writes example predictions to `artifacts/spam_example_predictions.csv`.

In [4]:
example_path = ARTIFACTS / "spam_example_predictions.csv"
if example_path.exists():
    display(pd.read_csv(example_path))
else:
    print("Run the training cell first to generate spam example predictions.")

,text,spam_probability,threshold,predicted_label
0,Get some free money,0.303406,0.41,ham
1,CONGRATULATIONS! You won a free prize! Click h...,0.971065,0.41,spam
2,"Hey, are we still meeting at 6 pm?",0.016902,0.41,ham
3,Thank you for helping our community organize t...,0.175410,0.41,ham
